# Gestion des Ruptures de Stock — Prévision & Alertes
## Étape 2 : Prévision avec Prophet (Time Series)

Dans cette deuxième partie, je m'attaque à la prévision. J'isole les produits les plus demandés, je formate leurs historiques de commandes pour Prophet, et je génère des prévisions pour les 30 prochains jours avec leurs intervalles de confiance. L'objectif est de dimensionner un stock de sécurité optimal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

data_path = "../data/Historical Product Demand.csv"
df = pd.read_csv(data_path)

# Nettoyage
df['Order_Demand'] = df['Order_Demand'].astype(str).str.replace(r'[^0-9-]', '', regex=True)
df['Order_Demand'] = pd.to_numeric(df['Order_Demand'], errors='coerce')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Order_Demand'])


### 1. Formatage pour Prophet
Je choisis le produit le plus demandé (Product_1359) pour démontrer la méthodologie de prévision. Prophet exige un format strict : une colonne `ds` (date) et une colonne `y` (valeur à prédire).

In [ ]:
target_product = 'Product_1359'
df_prod = df[df['Product_Code'] == target_product].copy()

# Agrégation journalière
df_daily = df_prod.groupby('Date')['Order_Demand'].sum().reset_index()

# Renommage pour Prophet
df_prophet = df_daily.rename(columns={'Date': 'ds', 'Order_Demand': 'y'})

print("Aperçu des données pour Prophet :")
print(df_prophet.tail())

### 2. Entraînement du modèle Prophet
Je lance l'entraînement sur l'historique et je demande une prévision sur 30 jours (horizon court terme).

In [ ]:
model = Prophet(interval_width=0.95, daily_seasonality=False)
model.fit(df_prophet)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

print("Prévisions pour les 5 derniers jours du mois à venir :")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

### 3. Calcul du Stock de Sécurité
Pour éviter la rupture, je ne me base pas sur la prévision moyenne (`yhat`), mais sur l'intervalle de confiance supérieur (`yhat_upper`).

In [ ]:
fig1 = model.plot(forecast)
plt.title(f"Prévision de la demande pour {target_product}")
plt.xlabel("Date")
plt.ylabel("Demande")
plt.show()

# Calcul sur les 30 jours futurs
future_forecast = forecast.tail(30)
total_expected_demand = future_forecast['yhat'].sum()
safety_stock = future_forecast['yhat_upper'].sum() - total_expected_demand

print(f"Demande totale prévue sur 30j : {total_expected_demand:,.0f} unités")
print(f"Stock de sécurité recommandé  : {safety_stock:,.0f} unités")
print(f"Stock minimum requis          : {total_expected_demand + safety_stock:,.0f} unités")

## Conclusions Business

1. **Anticipation des pics** : Grâce à Prophet, j'ai pu modéliser la tendance et la saisonnalité hebdomadaire/annuelle. Le modèle ne donne pas juste un chiffre plat, il suit les rythmes du marché.
2. **Gestion du risque via l'intervalle de confiance** : La vraie puissance de cette analyse réside dans la borne haute (`yhat_upper`). Le stock de sécurité n'est plus calculé au doigt mouillé, mais mathématiquement pour couvrir 95% des scénarios de pics de demande.
3. **Alertes de Rupture** : Si le stock physique actuel est inférieur au `Stock minimum requis` calculé, l'alerte de rupture de stock se déclenche immédiatement, permettant aux approvisionneurs de passer commande avec le bon délai.